In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.1 RoPE

Learned position embeddings are a lookup table capped at `block_size`. **Rotary**
position embeddings (RoPE) instead *rotate* the query and key vectors by an angle
proportional to position. Attention scores then depend on relative position, and
the model can extrapolate past its training length.

In [ ]:
from pocketlm import precompute_rope, apply_rope, PocketLM, PocketLMConfig

cos, sin = precompute_rope(head_size=8, seq_len=16)
x = torch.randn(1, 2, 5, 8)   # [batch, heads, tokens, head_size]
rotated = apply_rope(x, cos, sin)
print("norm preserved (rotation is length-preserving):",
      torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5))
model = PocketLM(PocketLMConfig(vocab_size=20, block_size=16, n_layer=2,
                                n_head=2, n_embd=32, pos="rope"))
print("rope model has a learned pos table:", hasattr(model, "pos_emb"))

RoPE adds no parameters and injects position *inside* attention, so a rope model
drops the learned position table entirely.

In [ ]:
assert torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5)
assert not hasattr(model, "pos_emb")